In [2]:
import os
import pickle
import pandas as pd
import numpy as np
from collections import Counter
from tqdm.auto import tqdm
import ast

# ================= 配置路径 =================
BASE_DIR = "/home/tyj/project/code_TangYuJie/processed_data/CIC-IDS-2017-session-srcIP_srcPort_dstIP_dstPort_proto"
INFO_PATH = os.path.join(BASE_DIR, "all_session_graph_info.pkl")
SESSION_SPLIT_CSV = os.path.join(BASE_DIR, "all_split_session.csv")
FLOW_CSV = os.path.join(BASE_DIR, "all_flow.csv")

def analyze_distribution():
    print(f"🔍 正在读取数据集信息: {BASE_DIR}")

    # 1. 加载图（Session）信息
    with open(INFO_PATH, 'rb') as f:
        info_data = pickle.load(f)

    label_names = info_data.get('label_name', [])
    label_ids = info_data.get('label_id', [])
    train_idx = info_data.get('train_index', [])
    val_idx = info_data.get('validate_index', [])
    test_idx = info_data.get('test_index', [])

    # 映射 ID 到名称
    unique_ids = sorted(list(set(label_ids)))
    id_to_name = {}
    for lid, name in zip(label_ids, label_names):
        if lid not in id_to_name:
            id_to_name[lid] = name

    # 统计图分布
    def count_graphs(indices):
        c = Counter([label_ids[i] for i in indices])
        return [c.get(lid, 0) for lid in unique_ids]

    graph_counts = {
        'train': count_graphs(train_idx),
        'val': count_graphs(val_idx),
        'test': count_graphs(test_idx)
    }

    # 2. 解析 Session 划分与 Flow UID 的对应关系
    print("⏳ 正在建立 Flow UID 到划分的映射 (Split Mapping)...")
    split_df = pd.read_csv(SESSION_SPLIT_CSV, usecols=['flow_uid_list', 'split'])
    uid_to_split = {}

    for _, row in tqdm(split_df.iterrows(), total=len(split_df), desc="解析 Session CSV"):
        split = row['split']
        uids = row['flow_uid_list']
        if isinstance(uids, str):
            uids = ast.literal_eval(uids)
        for uid in uids:
            uid_to_split[uid] = split

    # 3. 统计流（Flow）标签分布 (先用 raw label 统计)
    print("⏳ 正在读取 Flow CSV 统计流样本数量...")
    raw_flow_stats = {'train': Counter(), 'val': Counter(), 'test': Counter()}

    chunk_size = 500000
    for chunk in tqdm(pd.read_csv(FLOW_CSV, usecols=['uid', 'label'], chunksize=chunk_size), desc="扫描 Flow CSV"):
        chunk['label'] = chunk['label'].astype(str)
        # 批量处理该块
        for _, row in chunk.iterrows():
            split = uid_to_split.get(row['uid'])
            if split:
                s_key = 'val' if split == 'validate' else split
                raw_flow_stats[s_key][row['label']] += 1

    # ================= 核心修复：标签映射归类 =================
    flow_stats = {'train': Counter(), 'val': Counter(), 'test': Counter()}
    unmapped_labels = set()

    for s_key in ['train', 'val', 'test']:
        for raw_label, count in raw_flow_stats[s_key].items():
            mapped_name = None
            # 转为小写并替换部分符号，便于比较
            raw_norm = str(raw_label).lower().replace('_', ' ').replace('-', ' ')

            # 尝试与 id_to_name 中的标准名称进行匹配
            for lid, name in id_to_name.items():
                name_norm = name.lower().replace('_', ' ').replace('-', ' ')

                # 如果名称完全一致，或者图的名称包含在流的原始名称中 (例如 "web attack" 在 "web attack \xbf brute force" 中)
                if name_norm == raw_norm or name_norm in raw_norm:
                    mapped_name = name
                    break

            # 备选的无空格暴力匹配
            if not mapped_name:
                for lid, name in id_to_name.items():
                    if name.lower().replace(' ', '') in raw_norm.replace(' ', ''):
                        mapped_name = name
                        break

            if mapped_name:
                flow_stats[s_key][mapped_name] += count
            else:
                unmapped_labels.add(raw_label)
                flow_stats[s_key][raw_label] += count # 如果真的匹配不上，保留原名

    if unmapped_labels:
        print(f"\n⚠️ 警告: 有以下原始标签未能映射到你图(PKL)中的名称类别里: {unmapped_labels}")

    # 4. 汇总结果并打印表格
    print("\n" + "="*105)
    print(f"{'类别名称 (ID)':<25} | {'图(Train)':<10} | {'图(Val)':<10} | {'图(Test)':<10} || {'流(Train)':<10} | {'流(Val)':<10} | {'流(Test)':<10}")
    print("-" * 105)

    for i, lid in enumerate(unique_ids):
        name = id_to_name[lid]
        # 图数量
        g_t, g_v, g_te = graph_counts['train'][i], graph_counts['val'][i], graph_counts['test'][i]
        # 流数量 (现在由于映射，可以正确匹配上 name 了)
        f_t = flow_stats['train'].get(name, 0)
        f_v = flow_stats['val'].get(name, 0)
        f_te = flow_stats['test'].get(name, 0)

        print(f"{f'{name} ({lid})':<25} | {g_t:<10} | {g_v:<10} | {g_te:<10} || {f_t:<10} | {f_v:<10} | {f_te:<10}")

    # 计算总计
    print("-" * 105)
    sum_gt, sum_gv, sum_gte = len(train_idx), len(val_idx), len(test_idx)
    sum_ft = sum(flow_stats['train'].values())
    sum_fv = sum(flow_stats['val'].values())
    sum_fte = sum(flow_stats['test'].values())
    print(f"{'总计 (Total)':<25} | {sum_gt:<10} | {sum_gv:<10} | {sum_gte:<10} || {sum_ft:<10} | {sum_fv:<10} | {sum_fte:<10}")
    print("="*105)

if __name__ == "__main__":
    analyze_distribution()

🔍 正在读取数据集信息: /home/tyj/project/code_TangYuJie/processed_data/CIC-IDS-2017-session-srcIP_srcPort_dstIP_dstPort_proto
⏳ 正在建立 Flow UID 到划分的映射 (Split Mapping)...


解析 Session CSV:   0%|          | 0/636607 [00:00<?, ?it/s]

⏳ 正在读取 Flow CSV 统计流样本数量...


扫描 Flow CSV: 0it [00:00, ?it/s]


类别名称 (ID)                 | 图(Train)   | 图(Val)     | 图(Test)    || 流(Train)   | 流(Val)     | 流(Test)   
---------------------------------------------------------------------------------------------------------
benign (0)                | 425454     | 596        | 27688      || 437425     | 623        | 28449     
dos hulk (1)              | 162883     | 0          | 0          || 162883     | 0          | 0         
dos slowhttptest (2)      | 16045      | 0          | 0          || 16045      | 0          | 0         
dos slowloris (3)         | 3877       | 0          | 0          || 3877       | 0          | 0         
dos goldeneye (4)         | 7607       | 0          | 0          || 7607       | 0          | 0         
portscan (5)              | 31607      | 63360      | 64015      || 31607      | 63360      | 64015     
ddos (6)                  | 217        | 114        | 86135      || 217        | 114        | 86135     
ftp-patator (7)           | 3986       | 0          |

In [3]:
import dgl
import pandas as pd
import numpy as np
import os

# 1. 设置您的图数据路径
# 请根据您的实际生成路径替换，例如 'processed_data/cicids2017_graphs.bin'
graph_data_path = '/home/tyj/project/code_TangYuJie/processed_data/CIC-IDS-2017-session-srcIP_srcPort_dstIP_dstPort_proto/all_session_graph.bin'

print(f"Loading graphs from {graph_data_path}...")
try:
    # 加载 DGL 图数据
    graphs, label_dict = dgl.load_graphs(graph_data_path)

    # 2. 提取每个图的节点数和边数
    node_counts = [g.num_nodes() for g in graphs]
    edge_counts = [g.num_edges() for g in graphs]

    # 3. 构建 DataFrame
    df_stats = pd.DataFrame({
        'Node_Count': node_counts,
        'Edge_Count': edge_counts
    })

    # 4. 导出为 CSV 文件
    csv_filename = 'cicids2017_graph_stats.csv'
    df_stats.to_csv(csv_filename, index=False)

    print(f"Successfully processed {len(graphs)} graphs.")
    print(f"Data saved to: {os.path.abspath(csv_filename)}")

    # 打印前几行预览
    print("\nData Preview:")
    print(df_stats.head())

except Exception as e:
    print(f"Error loading graph data: {e}")
    print("请确保图文件路径正确且格式为 DGL 支持的 bin/pt 格式。")

Loading graphs from /home/tyj/project/code_TangYuJie/processed_data/CIC-IDS-2017-session-srcIP_srcPort_dstIP_dstPort_proto/all_session_graph.bin...
Successfully processed 899804 graphs.
Data saved to: /home/tyj/project/code_TangYuJie/src/analysis/cicids2017_graph_stats.csv

Data Preview:
   Node_Count  Edge_Count
0           1           1
1           1           1
2           1           1
3           1           1
4           1           1


In [ ]:
import os
import pickle
import pandas as pd
import numpy as np
from collections import Counter, defaultdict
from tqdm.auto import tqdm
import ast
import re

# ================= 配置路径 =================
BASE_DIR = "/home/disk_10T/tyj_data/processed_data/CIC-IoMT-2024-session-srcIP_srcPort_dstIP_dstPort_proto"
INFO_PATH = os.path.join(BASE_DIR, "all_session_graph_info.pkl")
SESSION_SPLIT_CSV = os.path.join(BASE_DIR, "all_split_session.csv")
FLOW_CSV = os.path.join(BASE_DIR, "all_flow.csv")

# 尝试导入Dask，如果没有安装则回退到Pandas（但强烈建议安装Dask）
try:
    import dask.dataframe as dd
    DASK_AVAILABLE = True
except ImportError:
    DASK_AVAILABLE = False
    print("⚠️  Dask 未安装，将使用 Pandas 单线程模式，速度较慢。")

def analyze_distribution_fast():
    print(f"🔍 正在读取数据集信息: {BASE_DIR}")

    # 1. 加载图（Session）信息 (这部分通常不大，保持原样)
    with open(INFO_PATH, 'rb') as f:
        info_data = pickle.load(f)

    label_names = info_data.get('label_name', [])
    label_ids = info_data.get('label_id', [])
    train_idx = info_data.get('train_index', [])
    val_idx = info_data.get('validate_index', [])
    test_idx = info_data.get('test_index', [])

    # 映射 ID 到名称
    unique_ids = sorted(list(set(label_ids)))
    id_to_name = {}
    for lid, name in zip(label_ids, label_names):
        if lid not in id_to_name:
            id_to_name[lid] = name

    # 统计图分布 (保持原样)
    def count_graphs(indices):
        c = Counter([label_ids[i] for i in indices])
        return [c.get(lid, 0) for lid in unique_ids]

    graph_counts = {
        'train': count_graphs(train_idx),
        'val': count_graphs(val_idx),
        'test': count_graphs(test_idx)
    }

    # ================= 优化点 2：极速构建 UID -> Split 映射 =================
    print("⏳ 正在建立 Flow UID 到划分的映射...")

    # 方法：使用 Pandas 原生操作替代 iterrows，速度提升 10-50 倍
    split_df = pd.read_csv(SESSION_SPLIT_CSV, usecols=['flow_uid_list', 'split'])

    # 安全地解析字符串列表 (如果已经是列表则跳过)
    print("   解析 Session 列表...")
    if isinstance(split_df['flow_uid_list'].iloc[0], str):
        split_df['flow_uid_list'] = split_df['flow_uid_list'].apply(ast.literal_eval)

    # 炸裂 (Explode) 列表，构建映射 DataFrame，然后转字典
    exploded = split_df.explode('flow_uid_list', ignore_index=True)
    # 重命名并确保类型一致
    exploded = exploded.rename(columns={'flow_uid_list': 'uid'})
    exploded['uid'] = exploded['uid'].astype(int) # 假设UID是整数

    # 构建映射字典：{uid: split}
    uid_to_split = dict(zip(exploded['uid'], exploded['split']))
    del split_df, exploded # 释放内存

    # ================= 优化点 3：构建智能标签映射器 =================
    print("⏳ 构建标签映射索引...")

    # 预处理：建立一个 "标准化关键词" -> "标准名称" 的映射
    # 同时构建一个正则表达式大模式，用于一次扫描匹配
    keyword_map = {}
    regex_patterns = []

    for lid, name in id_to_name.items():
        # 标准化标准名称 (小写，去符号)
        norm_name = name.lower().replace('_', ' ').replace('-', ' ')
        keyword_map[norm_name] = name

        # 构建正则：匹配完整词或包含关系
        # 使用 \b 词边界防止部分匹配错误，但为了兼容你的需求，我们做宽松匹配
        # 这里我们把标准名作为一个 pattern
        regex_patterns.append(re.escape(norm_name))

    # 按长度降序排序，优先匹配更长的名称 (例如 "Web Attack XSS" 优先于 "Web Attack")
    regex_patterns.sort(key=len, reverse=True)
    # 构建总正则
    master_regex = re.compile('|'.join(regex_patterns))

    # 定义映射函数
    def get_mapped_name(raw_label):
        raw_norm = str(raw_label).lower().replace('_', ' ').replace('-', ' ')

        # 1. 精确匹配
        if raw_norm in keyword_map:
            return keyword_map[raw_norm]

        # 2. 正则搜索 (一次查找即可，比双重循环快)
        match = master_regex.search(raw_norm)
        if match:
            return keyword_map[match.group(0)]

        # 3. 无空格暴力匹配 (作为备选)
        raw_norm_no_space = raw_norm.replace(' ', '')
        for pat, name in keyword_map.items():
            if pat.replace(' ', '') in raw_norm_no_space:
                return name

        return raw_label # 实在匹配不上返回原名

    # ================= 优化点 1：使用 Dask 并行处理 40G CSV =================
    print("⏳ 正在统计 Flow 分布 (使用 Dask 并行)...")

    # 初始化计数器
    flow_stats = {
        'train': defaultdict(int),
        'val': defaultdict(int),
        'test': defaultdict(int)
    }

    if DASK_AVAILABLE:
        # 使用 Dask 读取，指定 dtype 大幅减少内存
        # 假设 uid 是整数，label 是字符串
        ddf = dd.read_csv(
            FLOW_CSV,
            usecols=['uid', 'label'],
            dtype={'uid': 'int64', 'label': 'object'}
        )

        # 定义分区处理函数
        def process_partition(df):
            local_counts = {'train': Counter(), 'val': Counter(), 'test': Counter()}
            for _, row in df.iterrows():
                # 查找 Split
                split = uid_to_split.get(row['uid'])
                if not split:
                    continue

                s_key = 'val' if split == 'validate' else split

                # 映射 Label
                mapped_name = get_mapped_name(row['label'])
                local_counts[s_key][mapped_name] += 1
            return pd.Series(local_counts)

        # 执行 MapReduce
        print("   启动 Dask 计算集群...")
        # meta 告诉 Dask 输出格式
        meta = pd.Series(dtype='object', index=['train', 'val', 'test'])
        results = ddf.map_partitions(process_partition, meta=meta).compute()

        # 汇总结果
        print("   汇总各分区结果...")
        for res in results:
            for s_key in ['train', 'val', 'test']:
                for k, v in res[s_key].items():
                    flow_stats[s_key][k] += v

    else:
        # 回退方案：Pandas Chunk (但优化了内部逻辑)
        chunk_size = 500000
        for chunk in tqdm(pd.read_csv(FLOW_CSV, usecols=['uid', 'label'], chunksize=chunk_size, dtype={'uid': 'int64'}), desc="扫描 Flow CSV"):
            # 向量化操作先过滤出有 split 的 uid
            # 这里为了简单，还是用循环，但标签匹配已优化
            for _, row in chunk.iterrows():
                split = uid_to_split.get(row['uid'])
                if split:
                    s_key = 'val' if split == 'validate' else split
                    mapped_name = get_mapped_name(row['label'])
                    flow_stats[s_key][mapped_name] += 1

    # 转换回普通 Counter 以便后续处理
    flow_stats = {k: Counter(v) for k, v in flow_stats.items()}

    # ================= 输出结果 (保持原样) =================
    print("\n" + "="*105)
    print(f"{'类别名称 (ID)':<25} | {'图(Train)':<10} | {'图(Val)':<10} | {'图(Test)':<10} || {'流(Train)':<10} | {'流(Val)':<10} | {'流(Test)':<10}")
    print("-" * 105)

    for i, lid in enumerate(unique_ids):
        name = id_to_name[lid]
        g_t, g_v, g_te = graph_counts['train'][i], graph_counts['val'][i], graph_counts['test'][i]
        f_t = flow_stats['train'].get(name, 0)
        f_v = flow_stats['val'].get(name, 0)
        f_te = flow_stats['test'].get(name, 0)

        print(f"{f'{name} ({lid})':<25} | {g_t:<10} | {g_v:<10} | {g_te:<10} || {f_t:<10} | {f_v:<10} | {f_te:<10}")

    print("-" * 105)
    sum_gt, sum_gv, sum_gte = len(train_idx), len(val_idx), len(test_idx)
    sum_ft = sum(flow_stats['train'].values())
    sum_fv = sum(flow_stats['val'].values())
    sum_fte = sum(flow_stats['test'].values())
    print(f"{'总计 (Total)':<25} | {sum_gt:<10} | {sum_gv:<10} | {sum_gte:<10} || {sum_ft:<10} | {sum_fv:<10} | {sum_fte:<10}")
    print("="*105)

if __name__ == "__main__":
    analyze_distribution_fast()

In [ ]:
import dgl
import pandas as pd
import numpy as np
import os

# 1. 设置您的图数据路径
# 请根据您的实际生成路径替换，例如 'processed_data/cicids2017_graphs.bin'
graph_data_path = '/home/disk_10T/tyj_data/processed_data/CIC-IoMT-2024-session-srcIP_srcPort_dstIP_dstPort_proto/all_session_graph.bin'

print(f"Loading graphs from {graph_data_path}...")
try:
    # 加载 DGL 图数据
    graphs, label_dict = dgl.load_graphs(graph_data_path)

    # 2. 提取每个图的节点数和边数
    node_counts = [g.num_nodes() for g in graphs]
    edge_counts = [g.num_edges() for g in graphs]

    # 3. 构建 DataFrame
    df_stats = pd.DataFrame({
        'Node_Count': node_counts,
        'Edge_Count': edge_counts
    })

    # 4. 导出为 CSV 文件
    csv_filename = 'ciciomt2024_graph_stats.csv'
    df_stats.to_csv(csv_filename, index=False)

    print(f"Successfully processed {len(graphs)} graphs.")
    print(f"Data saved to: {os.path.abspath(csv_filename)}")

    # 打印前几行预览
    print("\nData Preview:")
    print(df_stats.head())

except Exception as e:
    print(f"Error loading graph data: {e}")
    print("请确保图文件路径正确且格式为 DGL 支持的 bin/pt 格式。")

In [1]:
import torch;
print(f"MKL可用: {torch.backends.mkl.is_available()}");
print(f"OpenMP可用: {torch.backends.openmp.is_available()}")

MKL可用: True
OpenMP可用: True


In [1]:
import dgl
import torch
import pprint
from dgl.data.utils import load_graphs, load_info

# 1. 定义你的生成文件路径
bin_path = "/home/tyj/project/code_TangYuJie/processed_data/CIC-IDS-2017-session-srcIP_srcPort_dstIP_dstPort_proto/all_session_graph.bin"
info_path = "/home/tyj/project/code_TangYuJie/processed_data/CIC-IDS-2017-session-srcIP_srcPort_dstIP_dstPort_proto/all_session_graph_info.pkl"

# 2. 加载数据
print("正在加载图数据，这可能需要几十秒...")
graphs, _ = load_graphs(bin_path)
info = load_info(info_path)

print(f"✅ 成功加载了 {len(graphs)} 张图。")
print("=" * 60)

# 3. 抽取第一张图进行宏观分析
g = graphs[0]
print(f"【图 0 宏观信息】")
print(f" - 节点数 (Flows): {g.num_nodes()}")
print(f" - 边数 (Relations): {g.num_edges()}")
print(f" - 标签 (Label): {info['label_name'][0]} (ID: {info['label_id'][0]})")
print(f" - 数据划分 (Split): {info['split'][0]}")
print(f" - 包含的节点特征键 (g.ndata.keys): {list(g.ndata.keys())}")
print("=" * 60)

# 4. 详细打印核心特征的张量结构 (查看第 0 个节点)
node_idx = 0
print(f"【节点 {node_idx} 的底层特征异构性分析】\n")

# 我们挑选最典型的三个异构特征来观察
features_to_check = ['flow_numeric_features', 'flow_categorical_features', 'packet_len_seq']

for feat_name in features_to_check:
    if feat_name in g.ndata:
        feat_tensor = g.ndata[feat_name]
        print(f"🔹 特征名称: {feat_name}")
        print(f"   - 全局张量形状 (Shape): {feat_tensor.shape}  <-- 注意这里的维度")
        print(f"   - 数据类型 (Dtype): {feat_tensor.dtype}")

        # 提取第 0 个节点的前 10 个元素展示
        sample_values = feat_tensor[node_idx][:10].tolist()

        # 格式化打印
        if feat_tensor.dtype in [torch.int64, torch.int32, torch.long]:
            print(f"   - 节点 {node_idx} 取值 (前10个): {[int(x) for x in sample_values]} ... (类别索引)")
        else:
            print(f"   - 节点 {node_idx} 取值 (前10个): {[round(float(x), 4) for x in sample_values]} ... (连续数值)")
        print("-" * 40)
    else:
        print(f"🔹 特征名称: {feat_name} (该图中未包含此特征)")
        print("-" * 40)

正在加载图数据，这可能需要几十秒...
✅ 成功加载了 899804 张图。
【图 0 宏观信息】
 - 节点数 (Flows): 1
 - 边数 (Relations): 1
 - 标签 (Label): benign (ID: 0)
 - 数据划分 (Split): train
 - 包含的节点特征键 (g.ndata.keys): ['id', 'flow_numeric_features', 'ssl_textual_attention_mask', 'dns_textual_attention_mask', 'burst_id', 'x509_textual_input_ids', 'domain_probs', 'ssl_textual_input_ids', 'dns_textual_input_ids', 'dns_numeric_features', 'packet_len_seq', 'ts', 'x509_textual_attention_mask', 'packet_iat_seq', 'packet_seq_mask']
【节点 0 的底层特征异构性分析】

🔹 特征名称: flow_numeric_features
   - 全局张量形状 (Shape): torch.Size([1, 88])  <-- 注意这里的维度
   - 数据类型 (Dtype): torch.float32
   - 节点 0 取值 (前10个): [-0.339, -0.0288, -0.0057, -0.0065, -0.0077, -0.0155, -0.0046, -0.0053, -0.339, -0.0077] ... (连续数值)
----------------------------------------
🔹 特征名称: flow_categorical_features (该图中未包含此特征)
----------------------------------------
🔹 特征名称: packet_len_seq
   - 全局张量形状 (Shape): torch.Size([1, 512])  <-- 注意这里的维度
   - 数据类型 (Dtype): torch.float32
   - 节点 0 取值 (前10个): [

In [2]:
# 导入必要库
import pandas as pd
import numpy as np
import os

# ====================== 第一步：定义过滤配置（仅保留攻击类别过滤） ======================
# 仅过滤以下8种攻击类别（严格按你的配置）
excluded_classes = [
    "malicious_Recon-Ping_Sweep",
    "malicious_MQTT-DoS-Publish_Flood",
    "malicious_MQTT-Malformed_Data",
    "malicious_MQTT-DDoS-Connect_Flood",
    "malicious_TCP_IP-DDoS-TCP",
    "malicious_TCP_IP-DoS-TCP",
    "malicious_Recon-Port_Scan",
    "malicious_ARP_Spoofing"
]

# ====================== 第二步：定义核心过滤函数（仅过滤攻击类别） ======================
def filter_iomt_data_only_classes(input_path, output_path):
    """
    仅过滤指定的攻击类别，不涉及端口/服务过滤
    :param input_path: 原始CSV文件路径（flow/session均可）
    :param output_path: 过滤后文件保存路径
    """
    # 分块大小（处理大文件，避免内存溢出）
    chunksize = 100000
    filtered_chunks = []

    # 逐块加载并过滤数据
    for chunk_idx, chunk in enumerate(pd.read_csv(input_path, chunksize=chunksize), 1):
        print(f"正在处理块 {chunk_idx}：原始行数 = {len(chunk)}")

        # 核心过滤：仅剔除指定的攻击类别
        if "label" not in chunk.columns:
            raise ValueError(f"文件 {input_path} 中缺少 'label' 列，请检查列名！")
        chunk_filtered = chunk[~chunk["label"].isin(excluded_classes)]
        print(f"块 {chunk_idx} 类别过滤后行数 = {len(chunk_filtered)}")

        filtered_chunks.append(chunk_filtered)

    # 合并所有块并保存
    filtered_data = pd.concat(filtered_chunks, ignore_index=True)
    filtered_data.to_csv(output_path, index=False, encoding="utf-8")

    # 打印过滤结果汇总
    print("\n========== 过滤完成汇总 ==========")
    print(f"原始文件：{input_path}")
    print(f"过滤后文件：{output_path}")
    print(f"最终保留总行数：{len(filtered_data)}")
    print(f"\n保留的类别分布（前20）：\n{filtered_data['label'].value_counts().head(20)}")
    return filtered_data

# ====================== 第三步：配置文件路径并执行过滤 ======================
# 请替换为你的实际文件路径！
flow_data_path = "/tmp/pycharm_project_982/processed_data/CIC-IoMT-2024-session-srcIP_srcPort_dstIP_dstPort_proto/all_flow.csv"
session_split_path = "/tmp/pycharm_project_982/processed_data/CIC-IoMT-2024-session-srcIP_srcPort_dstIP_dstPort_proto/all_session.csv"

# 过滤后文件保存目录（自动创建）
output_dir = "/tmp/pycharm_project_982/processed_data/filtered_data_only_classes"
os.makedirs(output_dir, exist_ok=True)
filtered_flow_path = os.path.join(output_dir, "filtered_all_flow.csv")
filtered_session_path = os.path.join(output_dir, "filtered_all_session.csv")

# 1. 过滤flow数据（仅过滤类别）
print("========== 开始过滤 Flow 数据（仅攻击类别） ==========")
filtered_flow = filter_iomt_data_only_classes(flow_data_path, filtered_flow_path)

# 2. 过滤session数据（仅过滤类别）
print("\n========== 开始过滤 Session 数据（仅攻击类别） ==========")
filtered_session = filter_iomt_data_only_classes(session_split_path, filtered_session_path)

# ====================== 第四步：验证过滤结果（仅校验类别） ======================
def verify_class_filter_result(filtered_data_path):
    """验证是否已彻底过滤指定的攻击类别"""
    # 读取部分数据快速校验
    sample_data = pd.read_csv(filtered_data_path, nrows=50000)
    # 检查是否存在排除的类别
    remaining_excluded = sample_data[sample_data["label"].isin(excluded_classes)]
    if len(remaining_excluded) > 0:
        print(f"⚠️  警告：{filtered_data_path} 中仍存在排除的攻击类别！")
        print(remaining_excluded["label"].value_counts())
    else:
        print(f"✅ {filtered_data_path} 类别过滤验证通过：无排除的攻击类别")

# 执行验证
print("\n========== 验证 Flow 数据过滤结果 ==========")
verify_class_filter_result(filtered_flow_path)
print("\n========== 验证 Session 数据过滤结果 ==========")
verify_class_filter_result(filtered_session_path)

========== 开始过滤 Flow 数据（仅攻击类别） ==========


/tmp/ipykernel_701314/141863012.py:31: DtypeWarning: Columns (0: dns.id.orig_h, 1: dns.id.resp_h, 2: dns.proto, 3: dns.query, 4: dns.qclass_name, 5: dns.qtype_name, 6: dns.rcode_name, 7: dns.AA, 8: dns.TC, 9: dns.RD, 10: dns.RA, 11: dns.answers, 12: dns.TTLs, 13: dns.rejected, 14: dns.cname_chain, 15: ssl.id.orig_h, 16: ssl.id.resp_h, 17: ssl.version, 18: ssl.cipher, 19: ssl.curve, 20: ssl.server_name, 21: ssl.resumed, 22: ssl.established, 23: ssl.cert_chain_fps, 24: ssl.client_cert_chain_fps, 25: ssl.next_protocol, 26: ssl.ssl_history, 27: ssl.sni_matches_cert, 28: x509.cert0.certificate.serial, 29: x509.cert0.certificate.subject, 30: x509.cert0.certificate.issuer, 31: x509.cert0.certificate.key_alg, 32: x509.cert0.certificate.sig_alg, 33: x509.cert0.certificate.key_type, 34: x509.cert0.certificate.curve, 35: x509.cert0.san.dns, 36: x509.cert0.basic_constraints.ca, 37: x509.cert1.certificate.serial, 38: x509.cert1.certificate.subject, 39: x509.cert1.certificate.issuer, 40: x509.cert1.

正在处理块 1：原始行数 = 100000
块 1 类别过滤后行数 = 55594
正在处理块 2：原始行数 = 100000
块 2 类别过滤后行数 = 0
正在处理块 3：原始行数 = 100000
块 3 类别过滤后行数 = 0
正在处理块 4：原始行数 = 100000
块 4 类别过滤后行数 = 0
正在处理块 5：原始行数 = 100000
块 5 类别过滤后行数 = 0


/tmp/ipykernel_701314/141863012.py:31: DtypeWarning: Columns (0: dns.id.orig_h, 1: dns.id.resp_h, 2: dns.proto, 3: dns.query, 4: dns.qclass_name, 5: dns.qtype_name, 6: dns.rcode_name, 7: dns.AA, 8: dns.TC, 9: dns.RD, 10: dns.RA, 11: dns.answers, 12: dns.TTLs, 13: dns.rejected, 14: dns.cname_chain, 15: http.id.orig_h, 16: http.id.resp_h, 17: http.method, 18: http.host, 19: http.uri, 20: http.status_msg, 21: http.tags) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk_idx, chunk in enumerate(pd.read_csv(input_path, chunksize=chunksize), 1):


正在处理块 6：原始行数 = 100000
块 6 类别过滤后行数 = 0


...

正在处理块 64：原始行数 = 100000
块 64 类别过滤后行数 = 19267


/tmp/ipykernel_701314/141863012.py:31: DtypeWarning: Columns (0: ssl.id.orig_h, 1: ssl.id.resp_h, 2: ssl.version, 3: ssl.cipher, 4: ssl.curve, 5: ssl.server_name, 6: ssl.resumed, 7: ssl.established, 8: ssl.cert_chain_fps, 9: ssl.client_cert_chain_fps, 10: ssl.ssl_history, 11: ssl.sni_matches_cert, 12: x509.cert0.certificate.serial, 13: x509.cert0.certificate.subject, 14: x509.cert0.certificate.issuer, 15: x509.cert0.certificate.key_alg, 16: x509.cert0.certificate.sig_alg, 17: x509.cert0.certificate.key_type, 18: x509.cert0.certificate.curve, 19: x509.cert0.san.dns, 20: x509.cert0.basic_constraints.ca, 21: x509.cert1.certificate.serial, 22: x509.cert1.certificate.subject, 23: x509.cert1.certificate.issuer, 24: x509.cert1.certificate.key_alg, 25: x509.cert1.certificate.sig_alg, 26: x509.cert1.certificate.key_type, 27: x509.cert1.certificate.curve, 28: x509.cert1.san.dns, 29: x509.cert1.basic_constraints.ca, 30: x509.cert2.certificate.serial, 31: x509.cert2.certificate.subject, 32: x509.c

正在处理块 65：原始行数 = 38903
块 65 类别过滤后行数 = 38903

========== 过滤完成汇总 ==========
原始文件：/tmp/pycharm_project_982/processed_data/CIC-IoMT-2024-session-srcIP_srcPort_dstIP_dstPort_proto/all_flow.csv
过滤后文件：/tmp/pycharm_project_982/processed_data/filtered_data_only_classes/filtered_all_flow.csv
最终保留总行数：514280

保留的类别分布（前20）：
label
malicious_TCP_IP-DDoS-UDP                   165723
malicious_MQTT-DoS-Connect_Flood            145529
malicious_TCP_IP-DoS-UDP                     58170
malicious_MQTT-DDoS-Publish_Flood            51075
malicious_Recon-OS_Scan                      30974
benign_active                                25010
benign_idle                                  14585
benign_unknown                               14125
malicious_Recon-VulScan                       4265
malicious_TCP_IP-DDoS-ICMP                    1812
malicious_TCP_IP-DoS-ICMP                     1138
benign_interaction_Ecobee_Camera               824
benign_power                                   389
benign_interaction_

/tmp/ipykernel_701314/141863012.py:31: DtypeWarning: Columns (0: ssl_version, 1: cipher_suite_server, 2: cert_key_alg, 3: cert_sig_alg, 4: cert_key_type) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk_idx, chunk in enumerate(pd.read_csv(input_path, chunksize=chunksize), 1):


ValueError: 文件 /tmp/pycharm_project_982/processed_data/CIC-IoMT-2024-session-srcIP_srcPort_dstIP_dstPort_proto/all_session.csv 中缺少 'label' 列，请检查列名！